# LangGraph：一个基于图结构的智能体框架

LangGraph 作为 LangChain 生态系统的重要扩展，代表了智能体框架设计的一个全新方向。与前面介绍的基于“对话”的框架（如 AutoGen 和 CAMEL）不同，LangGraph 将智能体的执行流程建模为一种状态机（State Machine），并将其表示为有向图（Directed Graph）。在这种范式中，图的节点（Nodes）代表一个具体的计算步骤（如调用 LLM、执行工具），而边（Edges）则定义了从一个节点到另一个节点的跳转逻辑。这种设计的革命性之处在于它天然支持循环，使得构建能够进行迭代、反思和自我修正的复杂智能体工作流变得前所未有的直观和简单。


要理解 LangGraph，首先我们要先掌握他的三个基本构成要素：

## 1.全局状态
在 LangGraph 中，全局状态（Global State）是一个共享的数据结构，所有节点都可以访问和修改它。这使得不同节点之间能够轻松地共享信息和上下文，从而实现更复杂的交互和协作。比如存储用户输入、LLM 输出、工具、对话历史调用结果等。

```python
from typing import TypedDict, List

# 定义全局状态的数据结构
class AgentState(TypedDict):
    messages: List[str]      # 对话历史
    current_task: str        # 当前任务
    final_answer: str        # 最终答案
    # ... 任何其他需要追踪的状态

```

## 2.节点（Nodes）
节点是 LangGraph 中的基本执行单元，每个节点代表一个具体的计算步骤。节点可以是调用 LLM、执行工具、进行条件判断等。每个节点都有一个 `run` 方法，接受全局状态作为输入，并返回一个新的状态。

```python
# 定义一个“规划者”节点函数
def planner_node(state: AgentState) -> AgentState:
    """根据当前任务制定计划，并更新状态。"""
    current_task = state["current_task"]
    # ... 调用LLM生成计划 ...
    plan = f"为任务 '{current_task}' 生成的计划..."

    # 将新消息追加到状态中
    state["messages"].append(plan)
    return state

# 定义一个“执行者”节点函数
def executor_node(state: AgentState) -> AgentState:
    """执行最新计划，并更新状态。"""
    latest_plan = state["messages"][-1]
    # ... 执行计划并获得结果 ...
    result = f"执行计划 '{latest_plan}' 的结果..."

    state["messages"].append(result)
    return state
```

## 3.边（Edges）

边定义了节点之间的跳转逻辑，决定了在什么条件下从一个节点跳转到另一个节点。这使得我们可以轻松地构建复杂的工作流，包括循环、条件分支等。

```python
def should_continue(state: AgentState) -> str:
    """条件函数：根据状态决定下一步路由。"""
    # 假设如果消息少于3条，则需要继续规划
    if len(state["messages"]) < 3:
        # 返回的字符串需要与添加条件边时定义的键匹配
        return "continue_to_planner"
    else:
        state["final_answer"] = state["messages"][-1]
        return "end_workflow"
```

## 整合到智能体中
最后，我们将这些节点和边整合到一个智能体中，定义整个工作流的执行逻辑。

```python
from langgraph.graph import StateGraph, END

# 初始化一个状态图，并绑定我们定义的状态结构
workflow = StateGraph(AgentState)

# 将节点函数添加到图中
workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)

# 设置图的入口点
workflow.set_entry_point("planner")

# 添加常规边，连接 planner 和 executor
workflow.add_edge("planner", "executor")

# 添加条件边，实现动态路由
workflow.add_conditional_edges(
    # 起始节点
    "executor",
    # 判断函数
    should_continue,
    # 路由映射：将判断函数的返回值映射到目标节点
    {
        "continue_to_planner": "planner", # 如果返回"continue_to_planner"，则跳回planner节点
        "end_workflow": END               # 如果返回"end_workflow"，则结束流程
    }
)

# 编译图，生成可执行的应用
app = workflow.compile()

# 运行图
inputs = {"current_task": "分析最近的AI行业新闻", "messages": []}
for event in app.stream(inputs):
    print(event)

```

## 基于LangGraph构三步问答助手

在理解了 LangGraph 的核心概念之后，我们将通过一个实战案例来巩固所学。我们将构建一个简化的问答对话助手，它会遵循一个清晰、固定的三步流程来回答用户的问题：

1. **理解问题**：首先，智能体会分析用户输入的问题，提取关键信息，并构建一个清晰的任务描述。
2. **搜索信息**：接下来，智能体会根据任务描述进行信息搜索，可能涉及调用外部工具或数据库来获取相关数据。
3. **生成答案**：最后，智能体会综合之前的步骤，生成一个准确、完整的答案，并返回给用户。


### 定义全局状态
我们首先定义一个全局状态的数据结构，用于在不同节点之间共享信息。


In [33]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class SearchState(TypedDict):
    messages: Annotated[list, add_messages]
    user_query: str      # 经过LLM理解后的用户需求总结
    search_query: str    # 优化后用于Tavily API的搜索查询
    search_results: str  # Tavily搜索返回的结果
    final_answer: str    # 最终生成的答案
    step: str            # 标记当前步骤


## 定义工作流节点
定义好状态结构后，下一步是创建构成我们工作流的各个节点。在 LangGraph 中，每个节点都是一个执行具体任务的 Python 函数。这些函数接收当前的状态对象作为输入，并返回一个包含更新后字段的字典。

在开始定义节点之前，我们先完成项目的初始化设置，包括加载环境变量和实例化大语言模型。

In [34]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from tavily import TavilyClient

# 加载 .env 文件中的环境变量
load_dotenv()

# 初始化模型
# 我们将使用这个 llm 实例来驱动所有节点的智能
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_ID", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL", "https://api.openai.com/v1"),
    temperature=0.7
)
# 初始化Tavily客户端
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


### 定义理解与查询节点

作为工作流第一步，我们需要一个节点来理解用户的查询，并将其转化为一个清晰的任务描述，并生成查询关键词。

In [35]:
def understand_query_node(state: SearchState) -> dict:
    """步骤1：理解用户查询并生成搜索关键词"""
    user_message = state["messages"][-1].content

    understand_prompt = f"""分析用户的查询："{user_message}"
请完成两个任务：
1. 简洁总结用户想要了解什么
2. 生成最适合搜索引擎的关键词（中英文均可，要精准）

格式：
理解：[用户需求总结]
搜索词：[最佳搜索关键词]"""

    response = llm.invoke([SystemMessage(content=understand_prompt)])
    response_text = response.content

    # 解析LLM的输出，提取搜索关键词
    search_query = user_message # 默认使用原始查询
    if "搜索词：" in response_text:
        search_query = response_text.split("搜索词：")[1].strip()

    return {
        "user_query": response_text,
        "search_query": search_query,
        "step": "understood",
        "messages": [AIMessage(content=f"我将为您搜索：{search_query}")]
    }


### 搜索节点

该节点负责执行智能体的“工具使用”能力，它将调用 Tavily API 进行真实的互联网搜索，并具备基础的错误处理功能。

In [36]:
def tavily_search_node(state: SearchState) -> SearchState:
    """步骤2：使用Tavily API进行真实搜索"""

    search_query = state["search_query"]

    try:
        print(f"🔍 正在搜索: {search_query}")

        # 调用Tavily搜索API
        response = tavily_client.search(
            query=search_query,
            search_depth="basic",
            include_answer=True,
            include_raw_content=False,
            max_results=5
        )

        # 处理搜索结果
        search_results = ""

        # 优先使用Tavily的综合答案
        if response.get("answer"):
            search_results = f"综合答案：\n{response['answer']}\n\n"

        # 添加具体的搜索结果
        if response.get("results"):
            search_results += "相关信息：\n"
            for i, result in enumerate(response["results"][:3], 1):
                title = result.get("title", "")
                content = result.get("content", "")
                url = result.get("url", "")
                search_results += f"{i}. {title}\n{content}\n来源：{url}\n\n"

        if not search_results:
            search_results = "抱歉，没有找到相关信息。"

        return {
            "search_results": search_results,
            "step": "searched",
            "messages": [AIMessage(content=f"✅ 搜索完成！找到了相关信息，正在为您整理答案...")]
        }

    except Exception as e:
        error_msg = f"搜索时发生错误: {str(e)}"
        print(f"❌ {error_msg}")

        return {
            "search_results": f"搜索失败：{error_msg}",
            "step": "search_failed",
            "messages": [AIMessage(content="❌ 搜索遇到问题，我将基于已有知识为您回答")]
        }

### 答案生成节点
最后的回答节点能够根据上一步的搜索是否成功，来选择不同的回答策略，具备了一定的弹性。

In [37]:
def generate_answer_node(state: SearchState) -> dict:
    """步骤3：基于搜索结果生成最终答案"""
    if state["step"] == "search_failed":
        # 如果搜索失败，执行回退策略，基于LLM自身知识回答
        fallback_prompt = f"搜索API暂时不可用，请基于您的知识回答用户的问题：\n用户问题：{state['user_query']}"
        response = llm.invoke([SystemMessage(content=fallback_prompt)])
    else:
        # 搜索成功，基于搜索结果生成答案
        answer_prompt = f"""基于以下搜索结果为用户提供完整、准确的答案：
用户问题：{state['user_query']}
搜索结果：\n{state['search_results']}
请综合搜索结果，提供准确、有用的回答..."""
        response = llm.invoke([SystemMessage(content=answer_prompt)])

    return {
        "final_answer": response.content,
        "step": "completed",
        "messages": [AIMessage(content=response.content)]
    }


### 构建图

In [38]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

def create_search_assistant():
    workflow = StateGraph(SearchState)

    # 添加节点
    workflow.add_node("understand", understand_query_node)
    workflow.add_node("search", tavily_search_node)
    workflow.add_node("answer", generate_answer_node)

    # 设置线性流程
    workflow.add_edge(START, "understand")
    workflow.add_edge("understand", "search")
    workflow.add_edge("search", "answer")
    workflow.add_edge("answer", END)

    # 编译图
    memory = InMemorySaver()
    app = workflow.compile(checkpointer=memory)
    return app


In [39]:
app = create_search_assistant()
query = "请帮我分析一下最近的AI行业新闻，尤其是关于大模型的最新进展。"
session_count = 0
session_count += 1
config = {"configurable": {"thread_id": f"search-session-{session_count}"}}

# 初始状态
initial_state = {
    "messages": [HumanMessage(content=query)],
    "user_query": "",
    "search_query": "",
    "search_results": "",
    "final_answer": "",
    "step": "start"
}

print("\n" + "="*60)

# 执行工作流
async for output in app.astream(initial_state, config=config):
    for node_name, node_output in output.items():
        if "messages" in node_output and node_output["messages"]:
            latest_message = node_output["messages"][-1]
            if isinstance(latest_message, AIMessage):
                if node_name == "understand":
                    print(f"🧠 理解阶段: {latest_message.content}")
                elif node_name == "search":
                    print(f"🔍 搜索阶段: {latest_message.content}")
                elif node_name == "answer":
                    print(f"\n💡 最终回答:\n{latest_message.content}")

print("\n" + "="*60 + "\n")



🧠 理解阶段: 我将为您搜索：AI行业最新新闻 大模型进展 2024 大语言模型动态 生成式AI新闻
🔍 正在搜索: AI行业最新新闻 大模型进展 2024 大语言模型动态 生成式AI新闻
🔍 搜索阶段: ✅ 搜索完成！找到了相关信息，正在为您整理答案...

💡 最终回答:
根据您提供的搜索结果，近期AI大模型领域的最新动态和进展主要集中在以下几个方面：

### 一、核心观点与行业反思
1.  **“扩展时代”面临瓶颈**：OpenAI联合创始人Ilya Sutskever等业界领袖指出，单纯依靠增加算力和数据规模的“扩展”发展模式正面临天花板。模型在测试中表现优异，但在实际应用中仍会犯低级错误，存在泛化缺陷。
2.  **行业存在“泡沫”与虚火**：有调查显示，约73%获得融资的AI初创公司，其产品实质是封装第三方API的“套壳”应用，技术宣传与实际落地存在巨大差距。同时，超过95%的企业AI试点项目失败，投入与产出存在“价值鸿沟”。

### 二、重要技术进展与产品动态
1.  **模型能力比拼**：
    *   **阿里千问**在空间推理基准测试中超越Gemini 3、GPT-5.1等国际顶尖模型，登顶全球冠军。
    *   **Claude Opus 4.5（推理版）与Sora Pro 2（推理版）** 等专注于复杂问题思考的推理模型发布，在智能指数和上下文窗口等方面展开竞争。
2.  **架构与效率优化**：
    *   **中兴通讯**发布论文，分析当前大模型（主要依赖Transformer架构）的发展瓶颈，探索下一代AI大模型的前沿方向。
    *   **中国联通**提出新的扩散模型加速方法，旨在解决视频生成速度慢、不稳定的问题。
    *   **亚马逊云科技**推出多实例GPU功能、双向流式传输技术，旨在最大化GPU利用率并实现实时、不间断的AI对话。
3.  **开发范式变革**：
    *   **谷歌**推出“文件搜索”等新功能，开发者仅需上传文件，模型即可自动完成信息检索，这被认为可能简化甚至“架空”传统的复杂RAG系统搭建流程。
    *   **阿里巴巴**开源智能体训练环境项目**ROCK**，与强化学习框架ROLL组合，解决智能体规模化训练难题。
4.  **图像生成模型**：德国黑森林实验室发布新一代图

## 总结

### 优势
- **清晰的结构**：LangGraph 的图结构使得智能体的执行流程非常清晰，每个节点的功能和责任都明确划分。
- **天然支持循环**：通过条件边，LangGraph 可以轻松实现迭代和自我修正的工作流，这对于构建复杂的智能体非常有用。
- **全局状态共享**：所有节点都可以访问和修改全局状态，使得不同节点之间的协作变得非常简单。

### 局限性

- **学习曲线**：对于习惯了传统对话式智能体设计的开发者来说，理解和使用图结构可能需要一定的学习和适应。
- **缺少动态能力** ：目前的设计更适合预定义的流程，对于需要动态生成节点或边的场景可能不太适用。